In [1]:
import os
from src.module.database import oracle_import, oracle_export, oracle_execute
#os.environ

In [2]:
marketplace_mp = oracle_import("""select id_, userid, p_date from toki.MONGO_MINIPROGRAMUSERLOGS partition(p_202605)
     where MINIPROGRAMID = '6821b668840548dbe178eacb'""")

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-07-23 10:52:10.581 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 1 min 20 sec 


In [3]:
marketplace_mp["USERID"].unique().shape

(258604,)

In [4]:
marketplace_mp.tail()

,ID_,USERID,P_DATE
633632,6a1c5078982236d956e8040d,65afc92e5687245adfba3f93,20260531
633633,6a1c516cf081a0899db9ed8a,66d5712c98ce1d7839f29b29,20260531
633634,6a1c516ec54f01072f707b4a,655f4c21e22c5f4c8b76d1ad,20260531
633635,6a1c51d12091a9e894b4d1b2,65d39d6735ef84b89ee1f638,20260531
633636,6a1c528e6adb6e8a91d03110,60d5415041f18afccfbdc833,20260531


In [5]:
consumer_events = oracle_import("select * from toki.marketplace_consumer_EVENTS where p_date >='20260701'")

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-07-23 10:52:39.451 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 20 sec 460 ms


In [6]:
customer_activities = oracle_import("select * from toki.marketplace_consumer_activities where p_date >='20260701'")

2026-07-23 10:54:19.229 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 1 min 39 sec 


In [7]:
customer_activities.head()

,ID_,ACTIVITYNAME,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
0,6a442142805151b025d9ff1d,cart-events,"{'cartId': '63b572a5119256f621a0c935', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
1,6a442142ce31add3c3475288,cart-events,"{'cartId': '5fb9457f58786d2fc4e4a6b5', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
2,6a442142ce31add3c347528a,cart-events,"{'cartId': '69a1cf749d7bf25a7dcdf8ad', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
3,6a442142ce31add3c347528c,cart-events,"{'cartId': '68d27a73ca4a9a5563b54e15', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
4,6a442142805151b025d9ff1f,cart-events,"{'cartId': '68d51739282d4349b9bd7681', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701


In [8]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '63b572a5119256f621a0c935', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, 'cart': {'_id': '69a1c0191449fddd6670fcc5', 'accountId': '63b572a5119256f621a0c935', 'items': [{'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, {'productId': '68febd519494859a95029a61', 'qty': 1, 'available': True, '_id': '69a54eac1449fddd66834326'}, {'productId': '68febd4c9494859a95029a58', 'qty': 1, 'available': True, '_id': '69a54eec1449fddd66834389'}], 'createdAt': '2026-02-27T16:02:33.965Z', 'updatedAt': '2026-06-30T20:04:18.220Z'}}"

In [9]:
import json
import ast

In [10]:
ast.literal_eval(customer_activities["ACTIVITYDATA"].values[0])

{'cartId': '63b572a5119256f621a0c935',
 'type': 'PRODUCT_MODIFIED',
 'item': {'productId': '692e313c49a5eecb319a3f58',
  'qty': 1,
  'available': True,
  '_id': '69a302381449fddd66765d91'},
 'cart': {'_id': '69a1c0191449fddd6670fcc5',
  'accountId': '63b572a5119256f621a0c935',
  'items': [{'productId': '692e313c49a5eecb319a3f58',
    'qty': 1,
    'available': True,
    '_id': '69a302381449fddd66765d91'},
   {'productId': '68febd519494859a95029a61',
    'qty': 1,
    'available': True,
    '_id': '69a54eac1449fddd66834326'},
   {'productId': '68febd4c9494859a95029a58',
    'qty': 1,
    'available': True,
    '_id': '69a54eec1449fddd66834389'}],
  'createdAt': '2026-02-27T16:02:33.965Z',
  'updatedAt': '2026-06-30T20:04:18.220Z'}}

In [11]:
from datetime import datetime, timedelta
snapshot_date = datetime.today().date()
snapshot_date = snapshot_date.strftime("%Y%m%d")

In [12]:
marketplace_products = oracle_import(f"select * from toki.marketplace_catalogue_products where SNAPSHOT_DATE >= to_date('{snapshot_date}', 'YYYYMMDD')")

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-07-23 10:56:05.867 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 1 min 46 sec 


In [13]:
marketplace_products.head()

,ID_,GROUPID,PRODUCTID,STOREID,SKU,BRAND,STOCK,MAINPRICE,SALEPRICE,SALEPERC,...,ASSETS,TAXONOMY,URL_,TAXON,VARIANTS,HASGIFT,CREATEDAT,UPDATEDAT,PRODUCTSTATE,SNAPSHOT_DATE
0,6a2fe95b38713901eb69c0e4,4d9f7925-afa2-59bd-9218-a7b57d6664c2,6a0546b4109dd372dc9c088d,691aef5c2dbd3a51d50796cb,ELEC-EMS-4560X,ELECTROLUX,1,2199900,1899900,13.64,...,['https://imagedelivery.net/jUCGGlEY6TCUtkJb-F...,"[{'id': '69fbf9771e4213029f4e7d31', 'level': 0...",/6a0546b4109dd372dc9c088d,"{'id': '69fbf9771e4213029f4e7d31', 'label': 'П...",[],False,2026-06-15 20:00:25,2026-06-24 13:34:23,SETTLED,2026-07-23 00:03:26
1,6a2fe95b38713901eb69c0e5,23d1f426-a04f-5c2e-8ae0-473c1a223846,6a0bfc8e174bffd07037ed16,691aef5c2dbd3a51d50796cb,MI-AF-6B,XIAOMI,7,339900,339900,0,...,['https://imagedelivery.net/jUCGGlEY6TCUtkJb-F...,"[{'id': '69fb0b5b9d0fc72657c65255', 'level': 0...",/6a0bfc8e174bffd07037ed16,"{'id': '69fb0b5b9d0fc72657c65255', 'label': 'Т...",[],False,2026-06-15 20:00:25,2026-07-22 20:00:21,ARCHIVED,2026-07-23 00:03:26
2,6a2fe95b38713901eb69c105,0b0b48aa-3c52-585e-a6a5-f88a2270dd8a,6a206bdf61436204972e0030,691aef5c2dbd3a51d50796cb,MI-APRO55-26,XIAOMI,30,1699900,1699900,0,...,['https://imagedelivery.net/jUCGGlEY6TCUtkJb-F...,"[{'id': '674425c2b07fff9ff4a48d4c', 'level': 0...",/6a206bdf61436204972e0030,"{'id': '674425c2b07fff9ff4a48d4c', 'label': 'З...",[],False,2026-06-15 20:00:25,2026-07-22 14:00:19,SETTLED,2026-07-23 00:03:26
3,6a2fe95b38713901eb69c106,ab715b94-fc5c-55ec-8d13-730ba04fd6ae,6a22661c87e3e71b6da449c0,691aef5c2dbd3a51d50796cb,KONK-KG80-12L24B,KONKA,209,949900,699900,26.32,...,['https://imagedelivery.net/jUCGGlEY6TCUtkJb-F...,"[{'id': '69fb11b38c9f6a3289e6437c', 'level': 0...",/6a22661c87e3e71b6da449c0,"{'id': '69fb11b38c9f6a3289e6437c', 'label': 'У...",[],False,2026-06-15 20:00:25,2026-07-22 14:00:19,SETTLED,2026-07-23 00:03:26
4,6a3a519738713901eb69c12d,8e603728-99d3-50f5-852a-9939faa1dd23,6a309f86d46aca65f80843c5,6a2f8804c7f1cc040b589a3b,ANTM-3784_19B3,DYSON,1,2850000,2850000,0,...,['https://new.antmall.mn/images/detailed/45/an...,"[{'id': '69fb16a36712683b9c3f5b84', 'level': 0...",/6a309f86d46aca65f80843c5,"{'id': '69fb16a36712683b9c3f5b84', 'label': 'Ү...","[{'productId': '6a309f86d46aca65f80843c5', '_i...",False,2026-06-23 17:27:51,2026-07-22 17:24:05,SETTLED,2026-07-23 00:03:26


In [14]:
productid = '69fc469bab34c8d11412ec79'
marketplace_products[marketplace_products["PRODUCTID"] == productid]

,ID_,GROUPID,PRODUCTID,STOREID,SKU,BRAND,STOCK,MAINPRICE,SALEPRICE,SALEPERC,...,ASSETS,TAXONOMY,URL_,TAXON,VARIANTS,HASGIFT,CREATEDAT,UPDATEDAT,PRODUCTSTATE,SNAPSHOT_DATE
329,6a0b3d3b38713901eb69bc30,86280094-3bd8-58e3-a377-b7f51bd157a2,69fc469bab34c8d11412ec79,6912d444458a4a2f9ca10060,SF321LF0,ROWENTA,1,199900,199900,0,...,['https://cdnp.cody.mn/spree/images/3431242/la...,"[{'id': '69fb16a36712683b9c3f5b84', 'level': 0...",/69fc469bab34c8d11412ec79,"{'id': '69fb16a36712683b9c3f5b84', 'label': 'Ү...","[{'productId': '69fc469bab34c8d11412ec79', '_i...",False,2026-05-19 00:24:26,2026-07-23 00:00:22,ARCHIVED,2026-07-23 00:03:26


In [15]:
consumer_events.shape

(302894, 11)

In [16]:
consumer_events.tail()

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
302889,6a60eaa9ce31add3c35884b2,taxon_click,{'taxon': {'label': 'Гал тогоо'}},6062bbf1855bbb3d19d02846,aMLgpECt-uBy_VgLZ2FH9czXYJxyspeg,2026-07-22T16:07:05.917Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like M...,https://marketplace.toki.mn/home/67442509f47e8...,2026-07-23 00:07:05,2026-07-23 00:07:05,20260723
302890,6a60eaaa4aeec353171e659e,taxon_click,{'taxon': {'label': 'Гал тогоо'}},6062bbf1855bbb3d19d02846,aMLgpECt-uBy_VgLZ2FH9czXYJxyspeg,2026-07-22T16:07:06.108Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like M...,https://marketplace.toki.mn/home/67442509f47e8...,2026-07-23 00:07:06,2026-07-23 00:07:06,20260723
302891,6a60eaaa4229463e57f43cd2,taxon_click,{'taxon': {'label': 'Гал тогоо'}},6062bbf1855bbb3d19d02846,aMLgpECt-uBy_VgLZ2FH9czXYJxyspeg,2026-07-22T16:07:06.311Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like M...,https://marketplace.toki.mn/home/67442509f47e8...,2026-07-23 00:07:06,2026-07-23 00:07:06,20260723
302892,6a60eaaa805151b025eb34d3,taxon_click,{'taxon': {'label': 'Гал тогоо'}},6062bbf1855bbb3d19d02846,aMLgpECt-uBy_VgLZ2FH9czXYJxyspeg,2026-07-22T16:07:06.483Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like M...,https://marketplace.toki.mn/home/67442509f47e8...,2026-07-23 00:07:06,2026-07-23 00:07:06,20260723
302893,6a60eaaa420fe633e03ed1e0,taxon_click,{'taxon': {'label': 'Гал тогоо'}},6062bbf1855bbb3d19d02846,aMLgpECt-uBy_VgLZ2FH9czXYJxyspeg,2026-07-22T16:07:06.668Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like M...,https://marketplace.toki.mn/home/67442509f47e8...,2026-07-23 00:07:06,2026-07-23 00:07:06,20260723


In [17]:
consumer_events.head(2)

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
0,6a44ce39ce31add3c347e3d6,product_click,"{'productIds': ['69fc469bab34c8d11412ec79'], '...",66fbc5824e022311128232ae,jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy,2026-07-01T08:21:23.894Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,https://marketplace.toki.mn/home/69f326e4f996d...,2026-07-01 16:22:17,2026-07-01 16:22:17,20260701
1,6a44ce3b420fe633e02e2e78,taxon_click,{'taxon': {'label': 'Гар утас'}},5ff870ee4f636263bd482270,2xI3rxpGJbBOeY4vnY_1EBSkTuAL8Pf9,2026-07-01T08:22:19.413Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_6 like M...,https://marketplace.toki.mn/home,2026-07-01 16:22:19,2026-07-01 16:22:19,20260701


In [18]:
consumer_events.groupby("EVENTNAME").count()

,ID_,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
EVENTNAME,,,,,,,,,,
product_click,133868,133868,133868,132798,133868,133868,133868,133868,133868,133868
taxon_click,169026,165120,169026,167699,169026,169026,169026,169026,169026,169026


In [34]:
consumer_events.head(2).to_json(orient = 'records')

/tmp/ipykernel_3013165/4066390533.py:1: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  consumer_events.head(2).to_json(orient = 'records')


'[{"ID_":"6a44ce39ce31add3c347e3d6","EVENTNAME":"product_click","EVENTVALUE":"{\'productIds\': [\'69fc469bab34c8d11412ec79\'], \'taxon\': {\'label\': \'\\u04ae\\u0441\\u043d\\u0438\\u0439 \\u0445\\u044d\\u0440\\u044d\\u0433\\u0441\\u044d\\u043b\'}}","ACCOUNTID":"66fbc5824e022311128232ae","SESSIONID":"jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy","TIMESTAMP_":"2026-07-01T08:21:23.894Z","USERAGENT":"Mozilla\\/5.0 (iPhone; CPU iPhone OS 18_7 like Mac OS X) AppleWebKit\\/605.1.15 (KHTML, like Gecko) Mobile\\/15E148","URL_":"https:\\/\\/marketplace.toki.mn\\/home\\/69f326e4f996d4fdbd2ade89\\/69fb16a36712683b9c3f5b84","CREATEDAT":1782922937000,"UPDATEDAT":1782922937000,"P_DATE":"20260701"},{"ID_":"6a44ce3b420fe633e02e2e78","EVENTNAME":"taxon_click","EVENTVALUE":"{\'taxon\': {\'label\': \'\\u0413\\u0430\\u0440 \\u0443\\u0442\\u0430\\u0441\'}}","ACCOUNTID":"5ff870ee4f636263bd482270","SESSIONID":"2xI3rxpGJbBOeY4vnY_1EBSkTuAL8Pf9","TIMESTAMP_":"2026-07-01T08:22:19.413Z","USERAGENT":"Mozilla\\/5.0 (iPhone; C

In [19]:
consumer_events[consumer_events["EVENTNAME"] == "taxon_click"]["EVENTVALUE"].unique()

<StringArray>
[                                                                                                                                '{'taxon': {'label': 'Гар утас'}}',
                                                                                                                                   '{'taxon': {'label': 'Зурагт'}}',
                                                                                                                          '{'taxon': {'label': 'Тренд технологи'}}',
                                                                                                                                '{'taxon': {'label': 'Гал тогоо'}}',
                                                                                                                                 '{'taxon': {'label': 'Гэр ахуй'}}',
                                                                                                                              '{'taxon': {'label': 'Мик, спикер'}

In [20]:
consumer_events[consumer_events["ACCOUNTID"] == '6a5e47214aeec353171ccaa0']

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE


In [21]:
consumer_events[consumer_events["SESSIONID"] == "jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy"]["EVENTVALUE"].values

<StringArray>
['{'productIds': ['69fc469bab34c8d11412ec79'], 'taxon': {'label': 'Үсний хэрэгсэл'}}',
 '{'productIds': ['6a309f8bd46aca65f808443d'], 'taxon': {'label': 'Үсний хэрэгсэл'}}',
 '{'productIds': ['6a052b7de66ad55426cacf70'], 'taxon': {'label': 'Үсний хэрэгсэл'}}',
                                                '{'taxon': {'label': 'Мик, спикер'}}',
         '{'productIds': ['69f9b680ce8b92727bb248a0'], 'taxon': {'label': 'Спикер'}}',
         '{'productIds': ['6a2bf50230ec49e38af99da7'], 'taxon': {'label': 'Спикер'}}',
                                             '{'taxon': {'label': 'Үсний хэрэгсэл'}}']
Length: 7, dtype: str

In [22]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '63b572a5119256f621a0c935', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, 'cart': {'_id': '69a1c0191449fddd6670fcc5', 'accountId': '63b572a5119256f621a0c935', 'items': [{'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, {'productId': '68febd519494859a95029a61', 'qty': 1, 'available': True, '_id': '69a54eac1449fddd66834326'}, {'productId': '68febd4c9494859a95029a58', 'qty': 1, 'available': True, '_id': '69a54eec1449fddd66834389'}], 'createdAt': '2026-02-27T16:02:33.965Z', 'updatedAt': '2026-06-30T20:04:18.220Z'}}"

In [23]:
customer_activities["ACTIVITYDATA"].values[2]

"{'cartId': '69a1cf749d7bf25a7dcdf8ad', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3e98b19bd8f5a236d7ee4e'}, 'cart': {'_id': '69a1d17e1449fddd6672c3e0', 'accountId': '69a1cf749d7bf25a7dcdf8ad', 'items': [{'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3e98b19bd8f5a236d7ee4e'}], 'createdAt': '2026-02-27T17:16:46.492Z', 'updatedAt': '2026-06-30T20:04:18.234Z'}}"

In [24]:
customer_activities["ACTIVITYNAME"].unique()

<StringArray>
['cart-events', 'limit-events', 'order-events', 'wishlist-events']
Length: 4, dtype: str

In [25]:
customer_activities.groupby(["ACTIVITYNAME"]).count()

,ID_,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
ACTIVITYNAME,,,,,
cart-events,2536718,2536718,2536718,2536718,2536718
limit-events,71346,71346,71346,71346,71346
order-events,13501,13501,13501,13501,13501
wishlist-events,1810,1810,1810,1810,1810


In [26]:
customer_activities[customer_activities["ACTIVITYNAME"]== 'cart-events'].head()["ACTIVITYDATA"].values[4]

"{'cartId': '68d51739282d4349b9bd7681', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3ab17ace94f86945bf6201'}, 'cart': {'_id': '69a250a11449fddd6672e579', 'accountId': '68d51739282d4349b9bd7681', 'items': [{'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3ab17ace94f86945bf6201'}], 'createdAt': '2026-02-28T02:19:13.501Z', 'updatedAt': '2026-06-30T20:04:18.250Z'}}"

In [27]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '63b572a5119256f621a0c935', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, 'cart': {'_id': '69a1c0191449fddd6670fcc5', 'accountId': '63b572a5119256f621a0c935', 'items': [{'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, {'productId': '68febd519494859a95029a61', 'qty': 1, 'available': True, '_id': '69a54eac1449fddd66834326'}, {'productId': '68febd4c9494859a95029a58', 'qty': 1, 'available': True, '_id': '69a54eec1449fddd66834389'}], 'createdAt': '2026-02-27T16:02:33.965Z', 'updatedAt': '2026-06-30T20:04:18.220Z'}}"

In [28]:
# there 4 events that could potentially be used to track user activity: "view_product"
# limit-events - user checked the lease limit
# order-events - user placed an order or completed an order
# wishlist-events - user added a product to their wishlist
# card-events added card or removed card, modified in the cards etc events,


In [29]:
from src.database import pgsql_import
master_catalog_profile = pgsql_import("select * from marketplace_catalog_data_extended_version3")

In [35]:
import pandas as pd
pd.set_option("display.max_columns", 100)

In [36]:
master_catalog_profile.head(2)

,carried_located_in,main_category,sub_category,product_category,exact_product_category,manufacturer,generic_name,actual_product,size,power_consumption,year,specifications,sku,connectivity,stock,price,dimensions,index,product_id,shop_name,discount,main_option,details,url_link,keywords,best_used_for,premium_grade,price_range,insurance,delivery,taxon_id,taxon_name,description,best_used_for_detail,taxondict,prompt_,created_at,colors,image_list,images,mainoption,data_relation_map,color,image,productstate,createdat,updatedat,productmeta,saleprice,image_urls,details_translation,group_id
0,Living Room,Electronics,Televisions,Mini LED QLED 4K TVs,Sony 85XR50 85-inch Mini LED QLED 4K HDR Googl...,SONY,Smart Television,Mini LED QLED 4K HDR Google 85 inch tv /SONY-K...,85 inches,,2026,"{""resolution"": ""UHD 4K 3840x2160p"", ""processor...",SONY-K-85XR50,"{""wifi"": ""Yes"", ""bluetooth"": ""Yes"", ""hdmi"": ""4...",2,11999900 MNT,Without stand: 1891 x 1085 x 492 mm; With stan...,shop_0_1772709420,6968a70895de95f954a9a16c,BSB Electronics,"{""regular_price"": ""12999900 MNT"", ""sale_price""...","{""screen-size"": ""85inch"", ""resolution"": ""3840x...",85-inch Sony Mini LED QLED 4K HDR Google TV wi...,https://imagedelivery.net/jUCGGlEY6TCUtkJb-Fl1...,"[""Sony TV"", ""85 inch TV"", ""Mini LED"", ""QLED"", ...",Entertainment,premium,luxury,,"Pickup, Shipping",674425c2b07fff9ff4a48d4c,tv,Screen size: 85 inch Resolution: UHD 4K 3840x2...,Optimized for home entertainment and streaming...,NaN,"{""role"": ""user"", ""content"": ""product index : 0...",2026-03-05 19:17:48.739441,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9769a9ad-9471-5347-abf4-c920d69a0e3e
1,Living Room,Electronics,Televisions,4K UHD TVs,Full Array LED 4K HDR Smart TV,Panasonic,Smart Television,Panasonic TH-75NX900,75 inch,,2026,"{""screen_type"": ""Full Array LED"", ""resolution""...",PANA-TH-75NX900M,"{""wifi"": ""Yes"", ""bluetooth"": ""Bluetooth 5.1"", ...",1,4599900 MNT,1675 x 1041 x 363 mm,shop_1_1772709420,6968a70895de95f954a9a16f,BSB Electronics,"{""regular_price"": ""5499900 MNT"", ""sale_price"":...","{""screen-size"": ""75 inch"", ""resolution"": ""3840...",PRODUCTSTATE: ARCHIVED; SYNCSTATE: SYNCED; CRE...,https://imagedelivery.net/jUCGGlEY6TCUtkJb-Fl1...,"[""Panasonic"", ""Panasonic TH-75NX900"", ""75 inch...",Entertainment,premium,high-end,,Pickup,674425c2b07fff9ff4a48d4c,tv,"Screen size: 75 inch, 1675mm Screen: Full Arra...",Ideal for home entertainment and streaming (mo...,NaN,"{""role"": ""user"", ""content"": ""product index : 1...",2026-03-05 19:18:43.058787,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eae2fd2c-1c51-56f4-a7ec-7561dbfd11e9


In [31]:
master_catalog_profile.tail(2).to_json(orient="records")

'[{"carried_located_in":"Home","main_category":"Home Appliances","sub_category":"Vacuum Cleaners","product_category":"Cordless Vacuum Cleaners","exact_product_category":"Dyson V15s Detect Submarine Complete","manufacturer":"DYSON","generic_name":"Vacuum Cleaner","actual_product":"Dyson V15s Detect Submarine\\u2122 Complete","size":"4.1 kg","power_consumption":"","year":"0","specifications":"{\\"suction_power\\": \\"240 AW\\", \\"runtime\\": \\"60 minutes\\", \\"charging_time\\": \\"4.5 hours\\", \\"filter\\": \\"Whole-machine HEPA\\", \\"cleaning_type\\": \\"dry and wet\\", \\"dust_detection\\": \\"Dust Detect sensor\\", \\"display\\": \\"LCD\\", \\"head_type\\": \\"Submarine wet roller head\\", \\"light\\": \\"Fluffy Optic laser\\"}","sku":"ANTM-3193","connectivity":"{}","stock":"4","price":"4100000 MNT","dimensions":"","index":"shop_49_1782115641","product_id":"6a309f8bd46aca65f8084431","shop_name":"ANTMALL","discount":"{}","main_option":"{}","details":"Cordless vacuum cleaner with w

In [32]:
import pandas as pd
pd.set_option('display.max_columns', 100)

In [33]:
print(master_catalog_profile.head(2).to_json(orient="records"))

[{"carried_located_in":"Living Room","main_category":"Electronics","sub_category":"Televisions","product_category":"Mini LED QLED 4K TVs","exact_product_category":"Sony 85XR50 85-inch Mini LED QLED 4K HDR Google TV","manufacturer":"SONY","generic_name":"Smart Television","actual_product":"Mini LED QLED 4K HDR Google 85 inch tv \/SONY-K-85XR50\/","size":"85 inches","power_consumption":"","year":"2026","specifications":"{\"resolution\": \"UHD 4K 3840x2160p\", \"processor\": \"XR Processor (image enhancement AI)\", \"panel_type\": \"Mini LED QLED\", \"wide_viewing\": \"X-Wide Angle\", \"anti_reflection\": \"X-Anti Reflection\", \"operating_system\": \"ANDROID (Google TV)\", \"preinstalled_apps\": [\"Netflix\", \"Prime Video\", \"Disney+\", \"YouTube\", \"Apple TV\"], \"ai_image_enhancement\": \"Yes\", \"refresh_rate\": \"Motionflow\u2122 XR 800Hz (Native 120Hz)\", \"eye_protection\": \"Yes\", \"audio\": [\"Dolby Atmos\", \"DSEE\", \"Cinema\", \"X-Balanced Speaker\"], \"hdmi_ports\": 4, \"

## Recommendation Engine — Live API Integration

The engine is running on **port 8018** (local) and publicly via Cloudflare Tunnel.

### Available endpoints
| Endpoint | Method | Purpose |
|---|---|---|
| `/api/v1/events` | POST | Ingest `customer_activities` events (order, cart, limit, wishlist, view, product_click, taxon_click) |
| `/api/v1/consumer-events` | POST | Ingest Oracle `consumer_events` rows directly (EVENTNAME, EVENTVALUE, ACCOUNTID …) |
| `/api/v1/infer` | POST | On-demand single-taxon inference for a user |
| `/api/v1/feed` | POST | Multi-taxon feed — returns top-N products per taxon based on interaction scores |
| `/api/v1/feed/push` | POST | Generate feed AND push it to the shop's API endpoint |
| `/api/v1/health` | GET | Liveness check |
| `/api/v1/catalog/status` | GET | Catalog index stats |

### Feed response format (shop receives)
```json
{
  "id": "<account_id>",
  "taxon_feeds": [
    { "taxon_id": "69fb126eae2e0da5c8bca3a0", "taxon_name": "household-appliances-multi-purpose-vacuum", "recommendations": ["pid1", "pid2", ...] },
    { "taxon_id": "69fbef9bda75a61ceadc7607", "taxon_name": "video-game-racing-wheel", "recommendations": ["pid3", "pid4", ...] }
  ],
  "total_products": 24,
  "strategy": "multi_taxon_hybrid",
  "intent_score": 9.5
}
```

In [ ]:
import requests, json

BASE = "http://localhost:8018"

# ── Catalog status ─────────────────────────────────────────────────────────────
status = requests.get(f"{BASE}/api/v1/catalog/status").json()
print(f"Catalog: {status['catalog_size']} products | TF-IDF {status['tfidf_shape']}")
print(f"Taxon labels mapped: {status['taxon_label_map_size']} | Taxon slugs: {status['taxon_name_map_size']}")
print(f"Active sessions: {status['active_sessions']} | Tracked users: {status['tracked_users']}")

In [ ]:
# ── POST /consumer-events: ingest real Oracle consumer_events rows ─────────────
# Simulates batch rows exactly as they appear in the Oracle table (uppercase field names)

sample_consumer_rows = [
    {
        "ID_": "6a44ce39ce31add3c347e3d6",
        "EVENTNAME": "product_click",
        "EVENTVALUE": "{'productIds': ['69fc469bab34c8d11412ec79'], 'taxon': {'label': '\u04ae\u0441\u043d\u0438\u0439 \u0445\u044d\u0440\u044d\u0433\u0441\u044d\u043b'}}",
        "ACCOUNTID": "66fbc5824e022311128232ae",
        "SESSIONID": "jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy",
        "TIMESTAMP_": "2026-07-01T08:21:23.894Z",
        "USERAGENT": "Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like Mac OS X) AppleWebKit/605.1.15",
    },
    {
        "ID_": "6a44ce3b420fe633e02e2e78",
        "EVENTNAME": "taxon_click",
        "EVENTVALUE": "{'taxon': {'label': '\u0413\u0430\u0440 \u0443\u0442\u0430\u0441'}}",
        "ACCOUNTID": "5ff870ee4f636263bd482270",
        "SESSIONID": "2xI3rxpGJbBOeY4vnY_1EBSkTuAL8Pf9",
        "TIMESTAMP_": "2026-07-01T08:22:19.413Z",
        "USERAGENT": "Mozilla/5.0 (iPhone; CPU iPhone OS 18_6 like Mac OS X) AppleWebKit/605.1.15",
    },
]

r = requests.post(f"{BASE}/api/v1/consumer-events", json={"events": sample_consumer_rows})
result = r.json()
print(f"Status: {result['status']} | processed={result['processed']} failed={result['failed']}")
print()
for rec in result["recommendations"]:
    print(f"  user: {rec['id']}")
    print(f"  taxon_id: {rec['taxon_id']}")
    print(f"  strategy: {rec['strategy']} | intent: {rec['intent_score']} | device: {rec['device']}")
    print(f"  recs ({rec['count']}): {rec['recommendations'][:4]}...")
    print()

In [ ]:
# ── POST /feed: multi-taxon recommendations ────────────────────────────────────
# Build up a user's history first, then generate a multi-taxon feed

# Simulate user history: viewed vacuum, ordered racing wheel, wishlisted Dyson
requests.post(f"{BASE}/api/v1/events", json={"events": [
    {"account_id": "demo_user_001", "activity_name": "product_click",
     "activity_data": {"productIds": ["69fc469bab34c8d11412ec79"], "taxon": {"label": "household-appliances-multi-purpose-vacuum"}}},
    {"account_id": "demo_user_001", "activity_name": "order-events",
     "activity_data": {"accountid": "demo_user_001", "productid": "698977503516dac1b3e97a6c", "action": "complete"}},
    {"account_id": "demo_user_001", "activity_name": "limit-events",
     "activity_data": {"accountid": "demo_user_001", "limit_amount": 3000000, "currency": "MNT"}},
    {"account_id": "demo_user_001", "activity_name": "wishlist-events",
     "activity_data": {"accountid": "demo_user_001", "productid": "6a309f8bd46aca65f8084431", "action": "add"}},
]})

# Now request multi-taxon feed
r = requests.post(f"{BASE}/api/v1/feed", json={
    "account_id": "demo_user_001",
    "top_taxons": 3,
    "top_n_per_taxon": 8,
})
feed = r.json()
print(f"Strategy: {feed['strategy']} | Intent: {feed['intent_score']} | Total products: {feed['total_products']}")
print()
for tf in feed["taxon_feeds"]:
    print(f"  [{tf['taxon_name']}]")
    print(f"   taxon_id: {tf['taxon_id']}")
    print(f"   recommendations ({tf['count']}): {tf['recommendations'][:4]}...")
    print()

In [ ]:
# ── POST /feed/push: push feed to shop's API ───────────────────────────────────
# Change `shop_feed_url` to your shop's actual endpoint.
# Push payload: {"id": account_id, "taxon_feeds": [{taxon_id, taxon_name, recommendations}]}

SHOP_FEED_URL = "https://your-shop.example.com/api/recommendations"  # replace with real URL

r = requests.post(f"{BASE}/api/v1/feed/push", json={
    "account_id": "demo_user_001",
    "top_taxons": 3,
    "top_n_per_taxon": 10,
    "shop_feed_url": SHOP_FEED_URL,  # set to None to use config.MARKETPLACE_API_BASE_URL
    "push_timeout_seconds": 3.0,
})
push_result = r.json()
print(f"Strategy: {push_result['strategy']} | Total products: {push_result['total_products']}")
print(f"Push status: {push_result['push_status']} | Push URL: {push_result['push_url']}")
if push_result["push_error"]:
    print(f"Push error: {push_result['push_error']}")
print()
for tf in push_result["taxon_feeds"]:
    print(f"  {tf['taxon_name'] or tf['taxon_id']}: {tf['count']} products")

In [ ]:
# ── Public URL (internet-exposed via Cloudflare Tunnel) ───────────────────────
# Share this URL with shop developers for testing
PUBLIC_URL = "https://alerts-anti-renaissance-harvey.trycloudflare.com"

print("=== Public API endpoints for shop developers ===")
print()
print(f"Docs/Swagger UI:   {PUBLIC_URL}/docs")
print(f"Health check:      GET  {PUBLIC_URL}/api/v1/health")
print(f"Catalog status:    GET  {PUBLIC_URL}/api/v1/catalog/status")
print()
print(f"Ingest events:     POST {PUBLIC_URL}/api/v1/events")
print(f"Consumer events:   POST {PUBLIC_URL}/api/v1/consumer-events")
print(f"Single-taxon infer:POST {PUBLIC_URL}/api/v1/infer")
print(f"Multi-taxon feed:  POST {PUBLIC_URL}/api/v1/feed")
print(f"Feed + push back:  POST {PUBLIC_URL}/api/v1/feed/push")
print()

# Quick health check via public URL
import requests
r = requests.get(f"{PUBLIC_URL}/api/v1/health", timeout=10)
print(f"Public health check: {r.status_code} | catalog_ready={r.json()['catalog_ready']}")